# 로컬 데이터 목록

`notebookf7f9c1f971.ipynb`는 Kaggle에서 같은 미세플라스틱 CSV를 읽어 MiMo-VL-7B를 LoRA로 맞추는 노트북이다. 여기서는 그 노트북의 데이터 읽기와 그림 부분만 가져오고, 경로를 로컬 `train/`, `valid/`로 바꿨다.

가져오지 않은 셀은 Gemini 설명 생성, 7B 모델 로드, LoRA 학습이다. 그 작업은 공식 분할 위의 비전-언어 모델 학습이라 현재 연구 단계와 다르다. 이 노트북의 산출물은 `artifacts/data_inventory.csv`다.

기준은 `../RESEARCH_PLAN.md`, 진행은 `../WORKLOG.md`의 T01이다.

In [ ]:
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd
from PIL import Image, ImageDraw


def find_root() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        train_csv = candidate / "train" / "_annotations.csv"
        valid_csv = candidate / "valid" / "_annotations.csv"
        if train_csv.exists() and valid_csv.exists():
            return candidate
    raise FileNotFoundError("train/valid _annotations.csv 를 찾지 못했다")


ROOT = find_root()
ART = ROOT / "artifacts"
ART.mkdir(parents=True, exist_ok=True)
ROOT

In [ ]:
def load_annotations(root: Path) -> pd.DataFrame:
    frames = []
    for split in ("train", "valid"):
        frame = pd.read_csv(root / split / "_annotations.csv")
        frame["split"] = split
        frames.append(frame)
    combined = pd.concat(frames, ignore_index=True)
    print("annotation rows", combined.groupby("split").size().to_dict())
    print("columns", combined.columns.tolist())
    print(combined["class"].value_counts())
    return combined


combined = load_annotations(ROOT)

In [ ]:
def parse_filename(filename: str) -> tuple[str, str]:
    stem = filename.split(".rf.")[0]
    if stem.endswith("_jpg"):
        stem = stem[: -len("_jpg")]
    if "--" in stem:
        prefix, number = stem.split("--", 1)
        return prefix, number.strip("-")
    return "", stem


def build_inventory(root: Path, annotations: pd.DataFrame) -> pd.DataFrame:
    rows = []
    names_in_csv = {}
    for split, frame in annotations.groupby("split"):
        names_in_csv[split] = set()
        for filename, group in frame.groupby("filename"):
            names_in_csv[split].add(filename)
            prefix, sample_id = parse_filename(filename)
            side_w = group["xmax"] - group["xmin"]
            side_h = group["ymax"] - group["ymin"]
            image_path = root / split / filename
            rows.append({
                "filename": filename,
                "split": split,
                "width": int(group["width"].iloc[0]),
                "height": int(group["height"].iloc[0]),
                "width_nunique": int(group["width"].nunique()),
                "height_nunique": int(group["height"].nunique()),
                "class_name": ",".join(sorted(group["class"].astype(str).unique())),
                "object_count": int(len(group)),
                "prefix": prefix,
                "sample_id": sample_id,
                "min_side_px": int(min(side_w.min(), side_h.min())),
                "file_on_disk": image_path.is_file(),
                "image_path": image_path.relative_to(root).as_posix(),
            })

    extra = []
    for split in ("train", "valid"):
        folder = root / split
        disk_names = {path.name for path in folder.iterdir() if path.suffix.lower() == ".jpg"}
        for filename in sorted(disk_names - names_in_csv.get(split, set())):
            prefix, sample_id = parse_filename(filename)
            extra.append({
                "filename": filename,
                "split": split,
                "width": pd.NA,
                "height": pd.NA,
                "width_nunique": 0,
                "height_nunique": 0,
                "class_name": "",
                "object_count": 0,
                "prefix": prefix,
                "sample_id": sample_id,
                "min_side_px": pd.NA,
                "file_on_disk": True,
                "image_path": (folder / filename).relative_to(root).as_posix(),
            })

    inventory = pd.DataFrame(rows)
    if extra:
        inventory = pd.concat([inventory, pd.DataFrame(extra)], ignore_index=True)
    inventory["csv_has_boxes"] = inventory["object_count"] > 0
    return inventory.sort_values(["split", "sample_id", "prefix", "filename"]).reset_index(drop=True)


inventory = build_inventory(ROOT, combined)
inventory_path = ART / "data_inventory.csv"
inventory.to_csv(inventory_path, index=False)
inventory_path, len(inventory)

In [ ]:
for split, frame in inventory.groupby("split"):
    missing = int((~frame["file_on_disk"]).sum())
    print(
        split,
        "images", len(frame),
        "boxes", int(frame["object_count"].sum()),
        "missing_file", missing,
        "classes", sorted(frame["class_name"].unique()),
    )

train_ids = set(inventory.loc[inventory["split"] == "train", "sample_id"])
valid_ids = set(inventory.loc[inventory["split"] == "valid", "sample_id"])
print("sample_id train", len(train_ids), "valid", len(valid_ids), "shared", len(train_ids & valid_ids))
print(inventory.groupby(["split", "prefix"]).size())
print("odd size rows")
print(inventory.loc[(inventory["width"] != 563) | (inventory["height"] != 537), ["filename", "split", "width", "height"]])

In [ ]:
plot_df = combined.copy()
plot_df["bbox_width"] = plot_df["xmax"] - plot_df["xmin"]
plot_df["bbox_height"] = plot_df["ymax"] - plot_df["ymin"]
plot_df["bbox_area"] = plot_df["bbox_width"] * plot_df["bbox_height"]
objects_per_image = plot_df.groupby(["filename", "split"]).size().reset_index(name="object_count")

fig, axes = plt.subplots(2, 2, figsize=(12, 9))
class_counts = plot_df["class"].value_counts()
axes[0, 0].bar(class_counts.index.astype(str), class_counts.values)
axes[0, 0].set_title("Class count")
axes[0, 1].hist(plot_df["bbox_area"], bins=50)
axes[0, 1].set_title("Bounding box area")
axes[0, 1].set_xlabel("pixels^2")
axes[1, 0].scatter(plot_df["width"], plot_df["height"], alpha=0.4)
axes[1, 0].set_title("Image size in annotations")
axes[1, 0].set_xlabel("width")
axes[1, 0].set_ylabel("height")
axes[1, 1].boxplot(
    [objects_per_image.loc[objects_per_image["split"] == split, "object_count"] for split in ("train", "valid")],
    tick_labels=["train", "valid"],
)
axes[1, 1].set_title("Objects per image")
fig.tight_layout()
overview_path = ART / "dataset_overview.png"
fig.savefig(overview_path, dpi=120)
plt.close(fig)

samples = []
for split in ("train", "valid"):
    names = sorted(inventory.loc[(inventory["split"] == split) & inventory["file_on_disk"], "filename"])
    samples.extend((split, name) for name in names[:2])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for axis, (split, filename) in zip(axes.ravel(), samples):
    image = Image.open(ROOT / split / filename).convert("RGB")
    draw = ImageDraw.Draw(image)
    boxes = combined[(combined["filename"] == filename) & (combined["split"] == split)]
    for _, box in boxes.iterrows():
        draw.rectangle([box["xmin"], box["ymin"], box["xmax"], box["ymax"]], outline="red", width=2)
    axis.imshow(image)
    axis.set_title(f"{split} / {filename}\n{len(boxes)} boxes")
    axis.axis("off")
fig.tight_layout()
sample_path = ART / "sample_boxes.png"
fig.savefig(sample_path, dpi=120)
plt.close(fig)
overview_path, sample_path